In [1]:
import polars as pl

df = pl.DataFrame(
    {
        "label": ["wd", "ws"],
        "data_1": [1, 1],
        "data_2": [2, 2],
        "start": [0, 1],
        "stop": [360, 20],
        "count": [1000, 1000],
    }
)
print(df)

shape: (2, 6)
┌───────┬────────┬────────┬───────┬──────┬───────┐
│ label ┆ data_1 ┆ data_2 ┆ start ┆ stop ┆ count │
│ ---   ┆ ---    ┆ ---    ┆ ---   ┆ ---  ┆ ---   │
│ str   ┆ i64    ┆ i64    ┆ i64   ┆ i64  ┆ i64   │
╞═══════╪════════╪════════╪═══════╪══════╪═══════╡
│ wd    ┆ 1      ┆ 2      ┆ 0     ┆ 360  ┆ 1000  │
│ ws    ┆ 1      ┆ 2      ┆ 1     ┆ 20   ┆ 1000  │
└───────┴────────┴────────┴───────┴──────┴───────┘


In [2]:
# dataframe with arrays for variable spaces
df_arr = df.select(
    pl.col("label"),
    pl.col("data_1"),
    pl.col("data_2"),
    var_1=pl.linear_spaces(pl.col("start"), pl.col("stop"), pl.col("count")),
)

print(df_arr)

shape: (2, 4)
┌───────┬────────┬────────┬─────────────────────────┐
│ label ┆ data_1 ┆ data_2 ┆ var_1                   │
│ ---   ┆ ---    ┆ ---    ┆ ---                     │
│ str   ┆ i64    ┆ i64    ┆ list[f64]               │
╞═══════╪════════╪════════╪═════════════════════════╡
│ wd    ┆ 1      ┆ 2      ┆ [0.0, 0.36036, … 360.0] │
│ ws    ┆ 1      ┆ 2      ┆ [1.0, 1.019019, … 20.0] │
└───────┴────────┴────────┴─────────────────────────┘


In [3]:
# exploded dataframe to include variable spaces in columnar data
df_1 = df_arr.explode("var_1")
print(df_1)

shape: (2_000, 4)
┌───────┬────────┬────────┬───────────┐
│ label ┆ data_1 ┆ data_2 ┆ var_1     │
│ ---   ┆ ---    ┆ ---    ┆ ---       │
│ str   ┆ i64    ┆ i64    ┆ f64       │
╞═══════╪════════╪════════╪═══════════╡
│ wd    ┆ 1      ┆ 2      ┆ 0.0       │
│ wd    ┆ 1      ┆ 2      ┆ 0.36036   │
│ wd    ┆ 1      ┆ 2      ┆ 0.720721  │
│ wd    ┆ 1      ┆ 2      ┆ 1.081081  │
│ wd    ┆ 1      ┆ 2      ┆ 1.441441  │
│ …     ┆ …      ┆ …      ┆ …         │
│ ws    ┆ 1      ┆ 2      ┆ 19.923924 │
│ ws    ┆ 1      ┆ 2      ┆ 19.942943 │
│ ws    ┆ 1      ┆ 2      ┆ 19.961962 │
│ ws    ┆ 1      ┆ 2      ┆ 19.980981 │
│ ws    ┆ 1      ┆ 2      ┆ 20.0      │
└───────┴────────┴────────┴───────────┘


In [4]:
# same thing to make a set of independent parameters
df_2 = (
    pl.DataFrame(
        {
            "option": ["opt_1", "opt_2", "opt_3"],
            "coef_1": [0.25, 0.5, 0.75],
            "coef_2": [1.5, 2, 2.5],
            "start": [1, 1, 1],
            "stop": [10, 10, 10],
            "count": [10000, 1000, 1000],
        }
    )
    .select(
        pl.col("option"),
        pl.col("coef_1"),
        pl.col("coef_2"),
        var_2=pl.linear_spaces(pl.col("start"), pl.col("stop"), pl.col("count")),
    )
    .explode("var_2")
)

print(df_2)

shape: (12_000, 4)
┌────────┬────────┬────────┬──────────┐
│ option ┆ coef_1 ┆ coef_2 ┆ var_2    │
│ ---    ┆ ---    ┆ ---    ┆ ---      │
│ str    ┆ f64    ┆ f64    ┆ f64      │
╞════════╪════════╪════════╪══════════╡
│ opt_1  ┆ 0.25   ┆ 1.5    ┆ 1.0      │
│ opt_1  ┆ 0.25   ┆ 1.5    ┆ 1.0009   │
│ opt_1  ┆ 0.25   ┆ 1.5    ┆ 1.0018   │
│ opt_1  ┆ 0.25   ┆ 1.5    ┆ 1.0027   │
│ opt_1  ┆ 0.25   ┆ 1.5    ┆ 1.0036   │
│ …      ┆ …      ┆ …      ┆ …        │
│ opt_3  ┆ 0.75   ┆ 2.5    ┆ 9.963964 │
│ opt_3  ┆ 0.75   ┆ 2.5    ┆ 9.972973 │
│ opt_3  ┆ 0.75   ┆ 2.5    ┆ 9.981982 │
│ opt_3  ┆ 0.75   ┆ 2.5    ┆ 9.990991 │
│ opt_3  ┆ 0.75   ┆ 2.5    ┆ 10.0     │
└────────┴────────┴────────┴──────────┘


In [5]:
# combine data sets
df_all = df_1.join(df_2, how="cross")
print(df_all)

shape: (24_000_000, 8)
┌───────┬────────┬────────┬───────┬────────┬────────┬────────┬──────────┐
│ label ┆ data_1 ┆ data_2 ┆ var_1 ┆ option ┆ coef_1 ┆ coef_2 ┆ var_2    │
│ ---   ┆ ---    ┆ ---    ┆ ---   ┆ ---    ┆ ---    ┆ ---    ┆ ---      │
│ str   ┆ i64    ┆ i64    ┆ f64   ┆ str    ┆ f64    ┆ f64    ┆ f64      │
╞═══════╪════════╪════════╪═══════╪════════╪════════╪════════╪══════════╡
│ wd    ┆ 1      ┆ 2      ┆ 0.0   ┆ opt_1  ┆ 0.25   ┆ 1.5    ┆ 1.0      │
│ wd    ┆ 1      ┆ 2      ┆ 0.0   ┆ opt_1  ┆ 0.25   ┆ 1.5    ┆ 1.0009   │
│ wd    ┆ 1      ┆ 2      ┆ 0.0   ┆ opt_1  ┆ 0.25   ┆ 1.5    ┆ 1.0018   │
│ wd    ┆ 1      ┆ 2      ┆ 0.0   ┆ opt_1  ┆ 0.25   ┆ 1.5    ┆ 1.0027   │
│ wd    ┆ 1      ┆ 2      ┆ 0.0   ┆ opt_1  ┆ 0.25   ┆ 1.5    ┆ 1.0036   │
│ …     ┆ …      ┆ …      ┆ …     ┆ …      ┆ …      ┆ …      ┆ …        │
│ ws    ┆ 1      ┆ 2      ┆ 20.0  ┆ opt_3  ┆ 0.75   ┆ 2.5    ┆ 9.963964 │
│ ws    ┆ 1      ┆ 2      ┆ 20.0  ┆ opt_3  ┆ 0.75   ┆ 2.5    ┆ 9.972973 │
│ ws    ┆ 1    

In [6]:
# calc results depending on vars

df_out = df_all.select(
    pl.col("option"),
    pl.col("var_1"),
    pl.col("var_2"),
    calc=pl.col("data_1")
    * pl.col("coef_1") ** pl.col("var_1")
    / pl.col("data_2")
    * pl.col("coef_2") ** pl.col("var_2"),
)
print(df_out)

shape: (24_000_000, 4)
┌────────┬───────┬──────────┬───────────┐
│ option ┆ var_1 ┆ var_2    ┆ calc      │
│ ---    ┆ ---   ┆ ---      ┆ ---       │
│ str    ┆ f64   ┆ f64      ┆ f64       │
╞════════╪═══════╪══════════╪═══════════╡
│ opt_1  ┆ 0.0   ┆ 1.0      ┆ 0.75      │
│ opt_1  ┆ 0.0   ┆ 1.0009   ┆ 0.750274  │
│ opt_1  ┆ 0.0   ┆ 1.0018   ┆ 0.750548  │
│ opt_1  ┆ 0.0   ┆ 1.0027   ┆ 0.750822  │
│ opt_1  ┆ 0.0   ┆ 1.0036   ┆ 0.751096  │
│ …      ┆ …     ┆ …        ┆ …         │
│ opt_3  ┆ 20.0  ┆ 9.963964 ┆ 14.630366 │
│ opt_3  ┆ 20.0  ┆ 9.972973 ┆ 14.751637 │
│ opt_3  ┆ 20.0  ┆ 9.981982 ┆ 14.873914 │
│ opt_3  ┆ 20.0  ┆ 9.990991 ┆ 14.997205 │
│ opt_3  ┆ 20.0  ┆ 10.0     ┆ 15.121517 │
└────────┴───────┴──────────┴───────────┘


In [7]:
print(df_out.group_by("option").agg(pl.col("calc").mean().name.suffix("_mean")))

shape: (3, 2)
┌────────┬───────────┐
│ option ┆ calc_mean │
│ ---    ┆ ---       │
│ str    ┆ f64       │
╞════════╪═══════════╡
│ opt_2  ┆ 1.752676  │
│ opt_1  ┆ 0.046757  │
│ opt_3  ┆ 42.629211 │
└────────┴───────────┘


# MARKDOWN TESTS

### Code block
```python
# comment
def foo():
    pass
```

### Flow chart
```mermaid
flowchart LR
    A[Start Node] --> B(Process Node)
    B --> C{Decision?}
    C -- Yes --> D[Success]
    C -- No --> E[Fail]
```